# Phase 1 Kaggle - MinerU/PDF-Extract-Kit First

Notebook n?y ch?y MinerU `pipeline` backend ?? tr?ch xu?t s?ch scan th?nh Markdown, metadata, ?nh/b?ng v? RAG chunks. DeepSeek-OCR/Qwen kh?ng b?t trong notebook n?y ?? tr?nh chi?m VRAM v? tr?nh dependency n?ng tr?n Kaggle. Sau khi MinerU ?n, c? th? ch?y DeepSeek ? notebook/server ri?ng ?? n?ng ch?t l??ng text OCR.

In [ ]:
from pathlib import Path
import json, math, os, re, shutil, subprocess, sys, time, zipfile
from datetime import datetime, timezone

# Kaggle c? s?n TensorFlow/Flax; ?p Transformers/MinerU ?i nh?nh PyTorch-only ?? tr?nh l?i pyOpenSSL/TensorFlow.
os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('TRANSFORMERS_NO_FLAX', '1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTHONUTF8', '1')
os.environ.setdefault('PYTHONIOENCODING', 'utf-8')

# Thay ???ng d?n input c?a file PDF v?o ??y.
INPUT_PDF = Path('/kaggle/input/vietnam-schoolbooks/SGK L?ch s? v? ??a l? 6 CD.pdf')

# N?u dataset Kaggle b? ??i t?n, t? t?m PDF ??u ti?n c? ?u?i .pdf trong /kaggle/input.
if not INPUT_PDF.exists() and Path('/kaggle/input').exists():
    candidates = sorted(Path('/kaggle/input').rglob('*.pdf'))
    if candidates:
        INPUT_PDF = candidates[0]

# Fallback khi ch?y local trong repo.
if not INPUT_PDF.exists():
    INPUT_PDF = Path('books/SGK L?ch s? v? ??a l? 6 CD.pdf')

OUTPUT_DIR = Path('/kaggle/working/class_6_mineru_phase1') if Path('/kaggle/working').exists() else Path('extracted/class_6_mineru_phase1')
MINERU_RAW_DIR = OUTPUT_DIR / '_mineru_raw'
PAGES_DIR = OUTPUT_DIR / 'pages'
IMAGES_DIR = OUTPUT_DIR / 'images'
METADATA_DIR = OUTPUT_DIR / 'metadata'
PREVIEW_DIR = OUTPUT_DIR / '_previews'

# Test nhanh tr??c v?i B?i 1. Khi ?n th? ??i START_PAGE=1, END_PAGE=None ?? ch?y to?n b? s?ch.
START_PAGE = 6      # 1-based
END_PAGE = 10       # None = ch?y h?t s?ch

# MinerU pipeline backend nh? h?n VLM/hybrid; kh?ng d?ng DeepSeek/Qwen trong notebook n?y.
MINERU_METHOD = 'ocr'
MINERU_BACKEND = 'pipeline'
MINERU_ENABLE_FORMULA = False
MINERU_ENABLE_TABLE = True
MINERU_PROCESSING_WINDOW_SIZE = 8
MINERU_RENDER_THREADS = 4
MINERU_TASK_TIMEOUT_SECONDS = 7200

# C?i nh?: th? pipeline/core tr??c. Kh?ng b?t all m?c ??nh v? all hay k?o vLLM/CUDA stack r?t n?ng.
MINERU_INSTALL_CANDIDATES = ['mineru[pipeline]', 'mineru[core]', 'mineru']
ALLOW_MINERU_ALL_FALLBACK = False

CHUNK_SIZE = 1800
CHUNK_OVERLAP = 200
ZIP_OUTPUT = True

for directory in [OUTPUT_DIR, MINERU_RAW_DIR, PAGES_DIR, IMAGES_DIR, METADATA_DIR, PREVIEW_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('INPUT_PDF =', INPUT_PDF)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('Page range =', START_PAGE, END_PAGE)


In [ ]:
def run_cmd(cmd, *, env=None, check=True):
    print('+', ' '.join(map(str, cmd)))
    sys.stdout.flush()
    return subprocess.run(list(map(str, cmd)), env=env, check=check)

run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip', 'uv'])
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pymupdf', 'pillow', 'python-docx', 'tqdm', 'numpy<2.5'])

installed_mineru = None
for package in MINERU_INSTALL_CANDIDATES:
    result = run_cmd(['uv', 'pip', 'install', '--system', '-q', '-U', package], check=False)
    if result.returncode == 0:
        installed_mineru = package
        break

if installed_mineru is None and ALLOW_MINERU_ALL_FALLBACK:
    result = run_cmd(['uv', 'pip', 'install', '--system', '-q', '-U', 'mineru[all]'], check=False)
    if result.returncode == 0:
        installed_mineru = 'mineru[all]'

if installed_mineru is None:
    raise RuntimeError('Cannot install MinerU. Try ALLOW_MINERU_ALL_FALLBACK=True or run this on a cleaner Kaggle image.')

# Gi? Transformers ? b?n ?? qua l?i PPDocLayoutV2 trong c?c l?n test tr??c.
run_cmd([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--force-reinstall', '--no-cache-dir',
    'transformers==4.57.1',
    'huggingface-hub>=0.34.0,<1.0',
    'tokenizers>=0.22,<0.23'
], check=False)

try:
    import torch
    GPU_COUNT = torch.cuda.device_count()
    GPU_IDS = list(range(GPU_COUNT))
except Exception:
    GPU_COUNT = 0
    GPU_IDS = []

CPU_COUNT = os.cpu_count() or 2
os.environ['OMP_NUM_THREADS'] = str(CPU_COUNT)
os.environ['MKL_NUM_THREADS'] = str(CPU_COUNT)
print('installed_mineru =', installed_mineru)
print('CPU_COUNT =', CPU_COUNT)
print('GPU_IDS =', GPU_IDS)
run_cmd(['mineru', '--help'], check=False)


In [ ]:
import fitz
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm


def render_one_page(args):
    pdf_path, page_index, scale, out_dir = args
    doc = fitz.open(str(pdf_path))
    page_no = page_index + 1
    out_path = Path(out_dir) / f'page_{page_no:03d}.jpg'
    if not out_path.exists():
        pix = doc[page_index].get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
        pix.save(str(out_path))
    return {'page_number': page_no, 'path': str(out_path)}

RENDER_SCALE = 2.0
pdf_doc = fitz.open(str(INPUT_PDF))
total_pages = len(pdf_doc)
start_idx = max(0, START_PAGE - 1)
end_idx = total_pages if END_PAGE is None else min(total_pages, END_PAGE)
page_indexes = list(range(start_idx, end_idx))
print('total_pages =', total_pages, 'selected =', len(page_indexes))

render_jobs = [(INPUT_PDF, i, RENDER_SCALE, PAGES_DIR) for i in page_indexes]
page_images = []
workers = min(CPU_COUNT, max(1, len(render_jobs)))
with ProcessPoolExecutor(max_workers=workers) as executor:
    futures = [executor.submit(render_one_page, job) for job in render_jobs]
    for future in tqdm(as_completed(futures), total=len(futures), desc='Render PDF pages'):
        page_images.append(future.result())
page_images = sorted(page_images, key=lambda item: item['page_number'])
print('rendered pages:', len(page_images))


In [ ]:
def mineru_env():
    env = os.environ.copy()
    env['USE_TF'] = '0'
    env['TRANSFORMERS_NO_TF'] = '1'
    env['USE_FLAX'] = '0'
    env['TRANSFORMERS_NO_FLAX'] = '1'
    env['TF_CPP_MIN_LOG_LEVEL'] = '3'
    env['MINERU_PROCESSING_WINDOW_SIZE'] = str(MINERU_PROCESSING_WINDOW_SIZE)
    env['MINERU_API_MAX_CONCURRENT_REQUESTS'] = '1'
    env['MINERU_PDF_RENDER_THREADS'] = str(min(MINERU_RENDER_THREADS, CPU_COUNT))
    env['MINERU_LOCAL_API_STARTUP_TIMEOUT_SECONDS'] = '600'
    env['MINERU_TASK_RESULT_TIMEOUT_SECONDS'] = str(MINERU_TASK_TIMEOUT_SECONDS)
    if GPU_IDS:
        env['CUDA_VISIBLE_DEVICES'] = ','.join(map(str, GPU_IDS))
    return env


def run_mineru():
    if not shutil.which('mineru'):
        raise RuntimeError('MinerU CLI not found after install.')
    cmd = [
        'mineru',
        '-p', str(INPUT_PDF),
        '-o', str(MINERU_RAW_DIR),
        '-b', MINERU_BACKEND,
        '-m', MINERU_METHOD,
        '-f', str(MINERU_ENABLE_FORMULA).lower(),
        '-t', str(MINERU_ENABLE_TABLE).lower(),
    ]
    if START_PAGE is not None:
        cmd += ['-s', str(max(0, START_PAGE - 1))]
    if END_PAGE is not None:
        cmd += ['-e', str(max(0, END_PAGE - 1))]
    else:
        print('Warning: END_PAGE=None, MinerU will parse the full PDF.')
    result = run_cmd(cmd, env=mineru_env(), check=False)
    return result.returncode == 0

mineru_ok = run_mineru()
print('mineru_ok =', mineru_ok)
if not mineru_ok:
    raise RuntimeError('MinerU failed. Check the traceback above before continuing.')


In [ ]:
def find_mineru_content_list(raw_dir):
    files = sorted(Path(raw_dir).rglob('*_content_list.json'))
    if not files:
        files = sorted(Path(raw_dir).rglob('content_list.json'))
    return files[0] if files else None


def find_mineru_markdown(raw_dir):
    candidates = []
    for path in sorted(Path(raw_dir).rglob('*.md')):
        name = path.name.lower()
        if 'origin' in name or 'debug' in name:
            continue
        candidates.append(path)
    return candidates[0] if candidates else None


def clean_text(value):
    return re.sub(r'\s+', ' ', str(value or '').replace('\u00a0', ' ')).strip()


def strip_accents(value):
    import unicodedata
    value = unicodedata.normalize('NFD', value)
    value = ''.join(ch for ch in value if unicodedata.category(ch) != 'Mn')
    return value.replace('?', 'd').replace('?', 'D')


def normalize_text(value):
    value = strip_accents(str(value)).lower().replace(',', '.').replace(':', '.')
    return re.sub(r'\s+', ' ', value).strip()


def figure_key(text):
    match = re.search(r'h(?:?|i)nh\s+(\d{1,2})\s*[\.,]\s*(\d{1,2})', str(text), flags=re.IGNORECASE)
    if not match:
        match = re.search(r'hinh\s+(\d{1,2})\s*[\.,]\s*(\d{1,2})', normalize_text(text))
    return f'{int(match.group(1))}_{int(match.group(2))}' if match else None


def copy_mineru_asset(content_path, src_rel, page_no, item_index, caption, item_type):
    if not src_rel:
        return ''
    src = content_path.parent / str(src_rel)
    if not src.exists():
        matches = list(Path(MINERU_RAW_DIR).rglob(Path(str(src_rel)).name))
        src = matches[0] if matches else src
    if not src.exists():
        return ''
    suffix = src.suffix or '.jpg'
    key = figure_key(caption)
    if key and item_type != 'table':
        stem = key
    elif key and item_type == 'table':
        stem = f'table_{key}'
    else:
        stem = f'mineru_p{page_no:03d}_{item_index:04d}'
    dst = IMAGES_DIR / f'{stem}{suffix}'
    dedupe = 2
    while dst.exists():
        dst = IMAGES_DIR / f'{stem}_{dedupe}{suffix}'
        dedupe += 1
    shutil.copy2(src, dst)
    return dst.relative_to(OUTPUT_DIR).as_posix()


def as_text_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return [str(x) for x in value if str(x).strip()]
    if isinstance(value, str):
        return [value] if value.strip() else []
    return [str(value)]


def item_caption(item):
    parts = []
    for key in ['image_caption', 'table_caption', 'chart_caption']:
        parts.extend(as_text_list(item.get(key)))
    return clean_text(' '.join(parts))


def item_text(item):
    parts = []
    for key in ['text', 'code_body', 'table_body']:
        if item.get(key):
            parts.append(str(item[key]))
    for key in ['image_caption', 'table_caption', 'chart_caption', 'image_footnote', 'table_footnote', 'chart_footnote']:
        parts.extend(as_text_list(item.get(key)))
    return clean_text(' '.join(parts))


def md_for_text(text, level=0):
    if not text:
        return []
    if isinstance(level, int) and level > 0:
        hashes = '#' * min(4, level + 1)
        return [f'{hashes} {text}', '']
    return [text, '']


def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        if end < len(text):
            punct = max(text.rfind('.', start, end), text.rfind('?', start, end), text.rfind('!', start, end))
            if punct > start + chunk_size * 0.55:
                end = punct + 1
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return [chunk for chunk in chunks if chunk]


content_path = find_mineru_content_list(MINERU_RAW_DIR)
markdown_path = find_mineru_markdown(MINERU_RAW_DIR)
print('content_path =', content_path)
print('markdown_path =', markdown_path)
if content_path is None:
    raise RuntimeError('No MinerU content_list.json found.')

content_items = json.loads(content_path.read_text(encoding='utf-8'))
page_blocks = {}
image_records = []
block_records = []
book_lines = [
    f'# {INPUT_PDF.stem}',
    '',
    f'> Source PDF: `{INPUT_PDF}`',
    f'> Generated at: `{datetime.now(timezone.utc).isoformat()}`',
    f'> Engine: `MinerU {MINERU_BACKEND}/{MINERU_METHOD}`',
    f'> Raw output: `{MINERU_RAW_DIR}`',
    '',
]

for idx, item in enumerate(content_items):
    item_type = item.get('type', 'text')
    page_no = int(item.get('page_idx', 0)) + 1
    page_blocks.setdefault(page_no, [])
    caption = item_caption(item)
    block = {
        'type': item_type,
        'order': len(page_blocks[page_no]),
        'page': page_no,
        'page_number': page_no,
        'bbox': item.get('bbox'),
        'source': 'mineru_pdf_extract_kit',
        'raw_type': item_type,
    }

    if item_type in {'image', 'chart', 'table'}:
        rel_path = copy_mineru_asset(content_path, item.get('img_path') or item.get('image_path'), page_no, idx, caption, item_type)
        visual_type = 'table' if item_type == 'table' else 'image'
        block.update({
            'type': visual_type,
            'id': figure_key(caption) or f'{visual_type}_{idx:04d}',
            'label': caption[:100] if caption else f'{visual_type} {idx}',
            'caption': caption,
            'path': rel_path,
        })
        image_records.append(block)
    else:
        text = item_text(item)
        block.update({
            'type': 'text',
            'text': text,
            'text_level': item.get('text_level', 0),
        })
    page_blocks[page_no].append(block)
    block_records.append(block)

rendered_page_numbers = [int(page['page_number']) for page in page_images] if page_images else []
raw_page_numbers = sorted(page_blocks)
if rendered_page_numbers and raw_page_numbers and not (set(rendered_page_numbers) & set(raw_page_numbers)) and len(rendered_page_numbers) == len(raw_page_numbers):
    page_remap = dict(zip(raw_page_numbers, rendered_page_numbers))
    remapped = {}
    for raw_page, blocks in page_blocks.items():
        new_page = page_remap.get(raw_page, raw_page)
        remapped[new_page] = blocks
        for block in blocks:
            block['page'] = new_page
            block['page_number'] = new_page
    page_blocks = remapped
    for block in block_records:
        raw_page = int(block.get('page_number') or block.get('page') or 0)
        if raw_page in page_remap:
            block['page'] = page_remap[raw_page]
            block['page_number'] = page_remap[raw_page]
    print('Remapped MinerU relative page_idx to PDF page numbers:', page_remap)

selected_page_numbers = rendered_page_numbers if rendered_page_numbers else sorted(page_blocks)
rag_records = []
page_records = []
for page_no in selected_page_numbers:
    blocks = page_blocks.get(page_no, [])
    book_lines += [f'## PDF Page {page_no}', '']
    text_parts = []
    images_for_page = []
    for block in blocks:
        if block.get('type') in {'image', 'table'}:
            if block.get('path'):
                alt = clean_text(block.get('caption') or block.get('label') or block.get('id'))
                book_lines += [f'![{alt}]({block["path"]})', '']
            if block.get('caption'):
                book_lines += [f'*{block["caption"]}*', '']
                text_parts.append(str(block['caption']))
            images_for_page.append({k: block.get(k) for k in ['id', 'path', 'caption', 'label', 'type']})
        else:
            text = clean_text(block.get('text'))
            text_parts.append(text)
            book_lines += md_for_text(text, block.get('text_level', 0))
    page_text = clean_text(' '.join(text_parts))
    page_record = {
        'source_pdf': str(INPUT_PDF),
        'page_number': page_no,
        'text': page_text,
        'text_blocks': blocks,
        'engine': 'mineru_pdf_extract_kit',
    }
    page_records.append(page_record)
    (METADATA_DIR / 'pages').mkdir(parents=True, exist_ok=True)
    (METADATA_DIR / 'pages' / f'page_{page_no:03d}.json').write_text(json.dumps(page_record, ensure_ascii=False, indent=2), encoding='utf-8')
    for chunk_idx, chunk in enumerate(chunk_text(page_text), start=1):
        rag_records.append({
            'chunk_id': f'page_{page_no:03d}_{chunk_idx:02d}',
            'source_pdf': str(INPUT_PDF),
            'page_number': page_no,
            'text': chunk,
            'images': images_for_page,
            'engine': 'mineru_pdf_extract_kit',
        })
    book_lines += ['</break>', '']

(OUTPUT_DIR / 'book.md').write_text('\n'.join(book_lines).strip() + '\n', encoding='utf-8')
(METADATA_DIR / 'images.json').write_text(json.dumps(image_records, ensure_ascii=False, indent=2), encoding='utf-8')
(METADATA_DIR / 'blocks.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in block_records) + '\n', encoding='utf-8')
(OUTPUT_DIR / 'rag_chunks.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in rag_records) + '\n', encoding='utf-8')
summary = {
    'source_pdf': str(INPUT_PDF),
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'page_range': {'start': START_PAGE, 'end': END_PAGE},
    'pages_processed': len(page_records),
    'mineru_raw_dir': str(MINERU_RAW_DIR),
    'mineru_content_list': str(content_path),
    'mineru_markdown': str(markdown_path) if markdown_path else None,
    'engine': {'layout': 'mineru_pdf_extract_kit', 'ocr': f'mineru_{MINERU_METHOD}'},
    'stats': {'blocks': len(block_records), 'images': len(image_records), 'rag_chunks': len(rag_records)},
    'outputs': {'markdown': 'book.md', 'docx': 'book.docx', 'rag_chunks': 'rag_chunks.jsonl', 'page_metadata_dir': 'metadata/pages', 'block_metadata': 'metadata/blocks.jsonl', 'image_metadata': 'metadata/images.json'},
}
(METADATA_DIR / 'book.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('Merged MinerU output:', OUTPUT_DIR)
print('pages:', len(page_records), 'images:', len(image_records), 'chunks:', len(rag_records))


In [ ]:
from PIL import Image, ImageDraw
from docx import Document
from docx.shared import Inches


def make_docx():
    book_md = OUTPUT_DIR / 'book.md'
    if not book_md.exists():
        return False
    doc = Document()
    doc.add_heading(INPUT_PDF.stem, level=1)
    for line in book_md.read_text(encoding='utf-8').splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith('> ') or stripped == '</break>':
            continue
        if stripped.startswith('# '):
            continue
        if stripped.startswith('## '):
            doc.add_heading(stripped[3:].strip(), level=2)
        elif stripped.startswith('### '):
            doc.add_heading(stripped[4:].strip(), level=3)
        elif stripped.startswith('#### '):
            doc.add_heading(stripped[5:].strip(), level=4)
        elif stripped.startswith('!['):
            match = re.search(r'\]\((.*?)\)', stripped)
            if match:
                image_path = OUTPUT_DIR / match.group(1)
                if image_path.exists():
                    try:
                        doc.add_picture(str(image_path), width=Inches(5.6))
                    except Exception:
                        doc.add_paragraph(str(image_path))
        else:
            paragraph = doc.add_paragraph(stripped)
            if stripped.startswith('*') and stripped.endswith('*'):
                for run in paragraph.runs:
                    run.italic = True
    doc.save(str(OUTPUT_DIR / 'book.docx'))
    return True


def make_contact_sheet():
    records = json.loads((METADATA_DIR / 'images.json').read_text(encoding='utf-8')) if (METADATA_DIR / 'images.json').exists() else []
    records = [r for r in records if r.get('path')]
    if not records:
        return None
    thumb_w, label_h, pad, cols = 260, 42, 12, 3
    rows = math.ceil(len(records) / cols)
    sheet = Image.new('RGB', (cols * thumb_w + (cols + 1) * pad, rows * (thumb_w + label_h) + (rows + 1) * pad), 'white')
    draw = ImageDraw.Draw(sheet)
    for idx, rec in enumerate(records):
        path = OUTPUT_DIR / rec['path']
        if not path.exists():
            continue
        with Image.open(path).convert('RGB') as img:
            img.thumbnail((thumb_w, thumb_w - label_h), Image.Resampling.LANCZOS)
            col, row = idx % cols, idx // cols
            x = pad + col * (thumb_w + pad)
            y = pad + row * (thumb_w + label_h + pad)
            sheet.paste(img, (x + (thumb_w - img.width) // 2, y))
            draw.text((x + 4, y + thumb_w - 12), f'{rec.get("id")} p.{rec.get("page_number") or rec.get("page")}', fill=(20, 20, 20))
    out = PREVIEW_DIR / 'images_contact.jpg'
    sheet.save(out, quality=88)
    return out

print('docx:', make_docx())
contact = make_contact_sheet()
print('contact:', contact)
for path in [OUTPUT_DIR / 'book.md', OUTPUT_DIR / 'book.docx', OUTPUT_DIR / 'rag_chunks.jsonl', METADATA_DIR / 'book.json', METADATA_DIR / 'images.json', contact]:
    if path and Path(path).exists():
        print(path, Path(path).stat().st_size)


In [ ]:
zip_path = OUTPUT_DIR.with_suffix('.zip')
if ZIP_OUTPUT:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for path in sorted(OUTPUT_DIR.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(OUTPUT_DIR.parent))
    print('zip:', zip_path, zip_path.stat().st_size)

try:
    from IPython.display import display, Image as IPImage
    contact = PREVIEW_DIR / 'images_contact.jpg'
    if contact.exists():
        display(IPImage(filename=str(contact)))
except Exception as exc:
    print(exc)

print('\nFinal output directory:')
print(OUTPUT_DIR)
print('\nImportant files:')
for rel in ['book.md', 'book.docx', 'rag_chunks.jsonl', 'metadata/book.json', 'metadata/images.json', '_previews/images_contact.jpg']:
    path = OUTPUT_DIR / rel
    print(rel, 'OK' if path.exists() else 'MISSING', path.stat().st_size if path.exists() else '')
